# Round 18: wider seed averaging + wider leaf diversity - S6E10

Two bounded experiments, one change each, on the strongest measured scheme (Exp 36), along
two previously productive
ensemble axes - seed averaging and leaf-count diversity (counts and leak-free TE were the
campaign's other measured wins and are already inside the scheme):

- **Exp 44: five-seed averaging.** Exp 36 exactly, one change: seeds {42,43,44} -> {42,43,44,
  45,46} (10 fits). Seed averaging measured positive twice (rounds 10 and 12); this tests whether more of it still helps, with diminishing returns expected.
- **Exp 45: wider leaf diversity.** Exp 36 exactly, one change: leaves {63,127} -> {63,127,
  255} (9 fits). Adding leaf-count diversity measured positive in round 12; this extends the
  set in the same direction.

Guard: refit fires only on a WORKING SIGNAL win vs the Exp 36 bar. 0.960 status at dispatch,
two facts: the preferred candidate was Exp 36 at 0.959996 (strictly below 0.960); the campaign's
highest measured number at that point (0.960018, round-15 blend scan) was an exploratory
adaptive same-OOF scan output, not a candidate score.


In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CROSS_PAIRS = [('Class', 'Type of Travel'), ('Customer Type', 'Class')]
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # the bar (seed-42 partition)


In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def cross_name(pair):
    return ' x '.join(pair)

CROSS_COLS = [cross_name(p) for p in CROSS_PAIRS]

def add_crosses(df):
    out = df.copy()
    for pair in CROSS_PAIRS:
        out[cross_name(pair)] = out[pair[0]].astype(str) + ' x ' + out[pair[1]].astype(str)
    return out

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding - not through the
    # value table and not through the smoothing mean (recomputed per inner fold from itr rows).
    # Inner folds are label-blind (plain shuffled KFold). (Round-10 audit-fixed construction.)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, te_drop=(),
             outer_seed=RANDOM_STATE, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)  # round 14: partition passed in
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
            if te_drop:  # raw crossed strings never enter the model - only their encodings
                X_fit = X_fit.drop(columns=list(te_drop))
                X_es = X_es.drop(columns=list(te_drop))
                X_va = X_va.drop(columns=list(te_drop))
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label, outer_seed=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=outer_seed)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features; crossed columns: {CROSS_COLS}')


def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label


Full train: 699,635 rows, 21 raw features; crossed columns: ['Class x Type of Travel', 'Customer Type x Class']


## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every seed-42-partition round.


In [3]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 1: AUC 0.95851 (best_iteration=296)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 2: AUC 0.95751 (best_iteration=265)


...

  INC fold 3: AUC 0.95907 (best_iteration=212)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 4: AUC 0.95841 (best_iteration=183)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  INC fold 5: AUC 0.95885 (best_iteration=555)
  determinism vs round-3 constants: max abs diff 0.00000 (PASS - identical folds)


## Exp 44: Exp 36 scheme, five seeds

Exp 36 exactly (counts + all-21 TE + stochastic, {63, 127} leaves), one change: the seed set
widens from {42,43,44} to {42,43,44,45,46} - 10 fits averaged per OOF row instead of 6.


In [4]:
SEEDS44 = [42, 43, 44, 45, 46]
LEAF_CONFIGS = [63, 127]
print(f'Exp 44: counts + all-{len(TE_COLS_ALL)} TE + stochastic, {len(SEEDS44)} seeds x {{63, 127}} leaves (Exp 36 used 3 seeds)')
oof_runs44, iters44 = {}, {}
for leaves in LEAF_CONFIGS:
    for seed in SEEDS44:
        o, a, it = oof_lgbm(features, target, f'E44-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH)
        oof_runs44[(leaves, seed)] = o
        iters44[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof44 = np.mean(list(oof_runs44.values()), axis=0)
aucs44 = fold_aucs(oof44, target)
print(f'Exp 44 (averaged) fold AUCs: {[round(a, 5) for a in aucs44]}')
print(f'Exp 44 mean OOF AUC: {np.mean(aucs44):.6f} +/- {np.std(aucs44):.5f} | pooled: {roc_auc_score(target, oof44):.6f}')


Exp 44: counts + all-21 TE + stochastic, 5 seeds x {63, 127} leaves (Exp 36 used 3 seeds)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E44-l63-s42 fold 1: AUC 0.95989 (best_iteration=284)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E44-l63-s42 fold 2: AUC 0.95864 (best_iteration=271)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E44-l127-s46 fold 4: AUC 0.95986 (best_iteration=256)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E44-l127-s46 fold 5: AUC 0.95984 (best_iteration=184)
  leaves=127 seed 46 mean AUC: 0.95974
Exp 44 (averaged) fold AUCs: [0.96015, 0.95885, 0.96069, 0.96012, 0.96019]


Exp 44 mean OOF AUC: 0.960001 +/- 0.00061 | pooled: 0.959998


## Exp 45: Exp 36 scheme, leaves {63, 127, 255}

Exp 36 exactly, one change: a third leaf count (255) joins {63, 127} - 9 fits averaged per
OOF row instead of 6. The round-12 win came from adding leaf-count diversity; this extends
the same axis one step.


In [5]:
LEAVES45 = [63, 127, 255]
SEEDS45 = [42, 43, 44]
print(f'Exp 45: counts + all-{len(TE_COLS_ALL)} TE + stochastic, 3 seeds x {{63, 127, 255}} leaves (Exp 36 used {{63, 127}})')
oof_runs45, iters45 = {}, {}
for leaves in LEAVES45:
    for seed in SEEDS45:
        o, a, it = oof_lgbm(features, target, f'E45-l{leaves}-s{seed}', seed=seed,
                            count_features=True, target_encode=True,
                            num_leaves=leaves, **STOCH)
        oof_runs45[(leaves, seed)] = o
        iters45[(leaves, seed)] = it
        print(f'  leaves={leaves} seed {seed} mean AUC: {np.mean(a):.5f}')
oof45 = np.mean(list(oof_runs45.values()), axis=0)
aucs45 = fold_aucs(oof45, target)
print(f'Exp 45 (averaged) fold AUCs: {[round(a, 5) for a in aucs45]}')
print(f'Exp 45 mean OOF AUC: {np.mean(aucs45):.6f} +/- {np.std(aucs45):.5f} | pooled: {roc_auc_score(target, oof45):.6f}')


Exp 45: counts + all-21 TE + stochastic, 3 seeds x {63, 127, 255} leaves (Exp 36 used {63, 127})


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E45-l63-s42 fold 1: AUC 0.95989 (best_iteration=284)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E45-l63-s42 fold 2: AUC 0.95864 (best_iteration=271)


...

/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E45-l255-s44 fold 4: AUC 0.95961 (best_iteration=172)


/Users/instinct/rep/kaggle/playground-series-s6e10/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


  E45-l255-s44 fold 5: AUC 0.95979 (best_iteration=183)
  leaves=255 seed 44 mean AUC: 0.95962
Exp 45 (averaged) fold AUCs: [0.96022, 0.95894, 0.96074, 0.96019, 0.96022]


Exp 45 mean OOF AUC: 0.960062 +/- 0.00060 | pooled: 0.960057


## Verdicts and refit guard

Each candidate pairs against the in-run incumbent, the Exp 29 folds, and the Exp 36 bar.
Refit fires only on a WORKING SIGNAL better vs the bar; on a tie or loss, Exp 36 and
submission_round12.csv stand. Bounded to the tested configurations.


In [6]:
results = {}
for tag, aucs, desc in [('44', aucs44, 'Exp 44: E36 scheme, 5 seeds x {63,127}'),
                        ('45', aucs45, 'Exp 45: E36 scheme, leaves {63,127,255}')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):          mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):      mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E36_FOLDS_R12)
    print(f'  vs Exp 36 folds (round 12 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 36 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 36 bar under the paired band. Exp 36 remains the '
          'preferred candidate; submission_round12.csv is untouched.')


Exp 44: E36 scheme, 5 seeds x {63,127}
  vs incumbent (in-run):          mean delta +0.00153 (spread 0.00016) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 29 folds (round 9):      mean delta +0.00067 (spread 0.00017) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 36 folds (round 12 bar): mean delta +0.00000 (spread 0.00002) - NO SIGNAL (within 2x per-fold delta spread - no action)
Exp 45: E36 scheme, leaves {63,127,255}
  vs incumbent (in-run):          mean delta +0.00159 (spread 0.00016) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 29 folds (round 9):      mean delta +0.00073 (spread 0.00018) - WORKING SIGNAL: better (descriptive, not proof)
  vs Exp 36 folds (round 12 bar): mean delta +0.00007 (spread 0.00002) - WORKING SIGNAL: better (descriptive, not proof)

REFIT GUARD: Exp 45 clears the Exp 36 bar under the paired band -> refit fires (exploratory, descriptive signal).


## Conditional refit (only if the guard fired)

Either branch: round-12 refit semantics on the Exp 36 frame (full-train counts + cross-fit TE,
3-or-5 seeds x the experiment's leaf set, n_estimators basis from the in-run seed-42
fold-mean best_iterations over the experiment's leaf set, logged).


In [7]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    leaves_set = LEAF_CONFIGS if win == '44' else LEAVES45
    seeds_set = SEEDS44 if win == '44' else SEEDS45
    iters_map = iters44 if win == '44' else iters45
    tag = ('Exp 44: E36 scheme, 5 seeds x {63,127}' if win == '44'
           else 'Exp 45: E36 scheme, leaves {63,127,255}')
    basis = int(round(np.mean([np.mean(iters_map[(l, 42)]) for l in leaves_set])))
    tr_c, (te_c,) = add_counts(features, [test_features])
    te_full = te_crossfit(tr_c, target, TE_COLS_ALL)
    te_test = te_apply(features, target, test_features, TE_COLS_ALL)
    X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
    X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
    X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
    preds = []
    for leaves in leaves_set:
        for seed in seeds_set:
            cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=basis,
                       n_jobs=-1, random_state=seed, verbose=-1)
            cfg.update(STOCH)
            model = lgb.LGBMClassifier(**cfg)
            model.fit(X_full, target)
            preds.append(model.predict_proba(X_test)[:, 1])
    p = np.mean(preds, axis=0)
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': p})
    sub.to_csv('submission_round18.csv', index=False)
    print(f'submission_round18.csv written: {len(sub):,} rows | {tag}, {len(leaves_set)*len(seeds_set)} fits, basis {basis}. '
          f'Local file only - NOT submitted; Andrea submits.')


submission_round18.csv written: 299,844 rows | Exp 45: E36 scheme, leaves {63,127,255}, 9 fits, basis 220. Local file only - NOT submitted; Andrea submits.


In [8]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round18_oof.npz',
         incumbent=oof_inc, exp44=oof44, exp45=oof45, target=target.to_numpy())
meta = {
    'round': 18,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'exp44': {'scheme': 'Exp 36 with 5 seeds (42-46) x {63,127} leaves, 10-fit average'},
        'exp45': {'scheme': 'Exp 36 with leaves {63,127,255} x 3 seeds, 9-fit average'},
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp44': aucs44, 'exp45': aucs45},
    'guard': {'winner': win, 'bar': 'E36_FOLDS_R12'},
}
(cache_dir / 'round18_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round18_oof.npz + round18_oof.json')


cached oof_cache/round18_oof.npz + round18_oof.json


## What happens next

- If Exp 44 or Exp 45 wins (bounded to that configuration): submission_round18.csv becomes the
  new local candidate.
- If both null: these two configurations are measured without a detected increment, bounded to
  themselves - not evidence about other untested directions. A run of null single-change tests
  around a stable candidate strengthens the plateau reading without proving a ceiling; the
  state of campaign is reported plainly, and external data (host-linked original dataset, CC0
  on its face, provenance caveats logged) remains Andrea's explicit choice on its own merits.
